In [1]:
import openai
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
openai.api_key = os.getenv('OPENAI_API_KEY')

def get_completion(messages, model="gpt-3.5-turbo", temperature=0):
    response = openai.ChatCompletion.create(
        model=model,
        messages=messages,
        temperature=temperature, # this is the degree of randomness of the model's output
    )
    return response.choices[0].message["content"]

system_role= {"role": "system", "content": """
    Act as a Software Developer who code using React and Node.js.    
"""}

In [5]:
repo_init_prompt = {"role": "user", "content": """
    Please help with the below tasks:
    - Node version to use and why?
    - Use app name: "timesheet-app"
    - Project setup
        - How to install node using nvm in macos
        - How to init the react project with typescript as the language
    - Suggest the Package.json changes for the below
        - Add dependencies
            - Use Istanbul for code coverage
                - Build failure on Code coverage threshold below 95%
                - Generate html report
            - Use Snyk for vulnerability scanning                
                - Set fail on high severity
            - ESLint for static code analysis
                - Build failure on error
                - Generate html report
        - Scripts for the below
            - coverage: to run the code coverage
            - lint: to run the static code analysis
            - snyk: to run the vulnerability scanning                
"""}

messages = [system_role, repo_init_prompt]
response = get_completion(messages)
print(response)

repo_response = {"role": "assistant", "content": response} 

Node version to use and why?
- It is recommended to use the latest LTS (Long Term Support) version of Node.js for production applications. As of now, the latest LTS version is Node.js 14.x. It provides better performance, stability, and security updates.

Use app name: "timesheet-app"
- Sure, we can use "timesheet-app" as the app name.

Project setup
- To install Node.js using nvm in macOS, follow the below steps:
    1. Install nvm (Node Version Manager) using the command: `curl -o- https://raw.githubusercontent.com/nvm-sh/nvm/v0.38.0/install.sh | bash`
    2. Close and reopen the terminal to start using nvm.
    3. Install the latest LTS version of Node.js using the command: `nvm install --lts`
    4. Verify the Node.js installation using the command: `node -v`

- To init the React project with TypeScript as the language, follow the below steps:
    1. Install create-react-app globally using the command: `npm install -g create-react-app`
    2. Create a new React project with TypeScr

In [3]:
git_ignore_prompt = {"role": "user", "content": """
    Suggest the .gitignore file for the below
    - React project
    - VS code
    - Intellij idea project files
    - Macos files
"""}

messages = [system_role, git_ignore_prompt]
response = get_completion(messages)
print(response)

Here are the suggested .gitignore files for each of the mentioned items:

React project:
```
# dependencies
/node_modules

# build output
/build
/dist

# IDE files
/.idea
*.iml

# misc
/.env
.DS_Store
```

VS Code:
```
.vscode/
*.code-workspace
```

Intellij idea project files:
```
# IntelliJ project files
.idea/
*.iml
*.ipr
*.iws
*.bak/
*.log
*.orig
```

MacOS files:
```
# macOS files
.DS_Store
.AppleDouble
.LSOverride
```


In [3]:
snyk_prompt = {"role": "user", "content": """
    Provide the below:
    - Install and add snyk, snyk-to-html as dev dependancy
    - npm script to run the snyk test and generate report in html format
    - How to fix vulnerabilities found by snyk
"""}

messages = [system_role, snyk_prompt]
response = get_completion(messages)
print(response)

To install and add snyk and snyk-to-html as dev dependencies, run the following command in your project directory:

```
npm install --save-dev snyk snyk-to-html
```

To run the snyk test and generate a report in HTML format, add the following script to your `package.json` file:

```
"scripts": {
  "snyk-test": "snyk test --json | snyk-to-html --output-file=snyk-report.html"
}
```

This script will run the snyk test and pipe the output to snyk-to-html, which will generate an HTML report and save it to a file named `snyk-report.html`.

To fix vulnerabilities found by snyk, you can follow the recommendations provided in the snyk report. Snyk will suggest specific versions of packages that you can upgrade to in order to fix the vulnerabilities. You can then update your `package.json` file with the new versions and run `npm install` to install the updated packages. If there are no available updates, you may need to consider alternative packages or implement additional security measures to m

In [5]:
messages = [system_role,  
{"role": "user", "content": """
    Using these requirements
    - S3 bucket name: "timesheet-app-s3-bucket"
    - Frontend code base location: "frontend/timesheet-app"

    Provide the Github action workflow code to do the below:
    - build the react project
    - use aws-actions/aws-cli to do the below
        - Sync the build folder to the s3 bucket
        - Get the cloudfront distribution id with Tag "Project" = "aus-generative-ai-experiment"
        - Update the cloudfront distribution with the above id after the deployment
"""}]

response = get_completion(messages)
print(response)

Here's the Github action workflow code to achieve the requirements:

```yaml
name: Deploy Timesheet App

on:
  push:
    branches:
      - main

jobs:
  deploy:
    runs-on: ubuntu-latest

    steps:
      - name: Checkout code
        uses: actions/checkout@v2

      - name: Install dependencies
        run: npm install
        working-directory: frontend/timesheet-app

      - name: Build React project
        run: npm run build
        working-directory: frontend/timesheet-app

      - name: Configure AWS credentials
        uses: aws-actions/configure-aws-credentials@v1
        with:
          aws-access-key-id: ${{ secrets.AWS_ACCESS_KEY_ID }}
          aws-secret-access-key: ${{ secrets.AWS_SECRET_ACCESS_KEY }}
          aws-region: us-east-1

      - name: Sync build folder to S3 bucket
        run: aws s3 sync build s3://timesheet-app-s3-bucket
        working-directory: frontend/timesheet-app

      - name: Get CloudFront distribution ID
        id: get_distribution_id
       